# Predicting the p-factor (and its dimensions) from demographics + structural + functional MRI

**NeuroHackademy 2026, week-1 group project.** One self-contained, end-to-end notebook that runs locally and on
JupyterHub. It searches a grid of **feature tiers x machine-learning algorithms** with leakage-free nested
cross-validation, finds the configuration that maximizes held-out R-squared, then applies the winner to the
p-factor and its three sub-dimensions (internalizing, externalizing, attention).

**Modalities and namespaces**
- `demo:*` demographics (age, sex, race, ethnicity, handedness, educations, BMI)
- `global:*` global structural MRI (eTIV, tissue-class volume fractions, mean thickness, pial area)
- `subcort:*` subcortical volumes (aseg): bilateral mean fraction and left-right asymmetry
- `cortex:aparc:*` Desikan-Killiany cortical morphometry (34 regions/hemisphere: thickness, area, gray volume)
- `fc:cwithin:* / fc:cbetween:*` cortical within / between Yeo-7 network functional connectivity (rest)
- `fc:swithin:* / fc:s2ctx:* / fc:s2net:*` subcortical within / to-cortex / to-network functional connectivity
- `qc:euler` FreeSurfer QC (never a predictor); `target*` = p-factor and its dimensions

### An honest, evidence-based expectation for R-squared (read this)
The p-factor is only weakly predictable from brain data. Across the literature, **out-of-sample** prediction of a
general-psychopathology factor from neuroimaging sits at **R-squared roughly 0.02 to 0.15** (see
[Xia et al. 2018, PNC p-factor + connectivity](https://www.nature.com/articles/s41467-018-05317-y); multimodal
out-of-sample predictions are weakly correlated with symptoms, r-squared < 0.15). This notebook, after an exhaustive
tier x model search with target-transform and stacking variants, reaches **R-squared about 0.08** for the p-factor,
which is a **predicted-observed correlation r about 0.28**. That r (0.2 to 0.35) is exactly what published PNC papers
report; many headline the correlation rather than R-squared. A value above ~0.30 R-squared is **not attainable** from
these features out of sample, and any pipeline that shows it is leaking or overfitting. The deliverable here is a
rigorous, reproducible estimate of a small but real effect, reported with both R-squared and r.

Negative R-squared is possible and legitimate: it means a model predicts worse than the training mean on held-out
data (overfitting). Well-regularized models avoid it; the leaderboard below shows negatives only for poorly matched
tier x model pairs, which the search then rejects.


## How to run

Select the **Python (nh2026)** kernel (or any Python whose NumPy stack is consistent) and run top to bottom. The first
cell verifies the environment and stops with a clear fix if the kernel has a broken NumPy (the classic
`numpy.dtype size changed` error means the wrong interpreter; switch kernel or `pip install 'numpy<2'`).

The leaderboard cell is the slowest (about **5 to 10 minutes**; set `FAST_MODE = True` in the config cell for a
~2-minute pass). Everything is cached to a single dataset CSV, so re-runs and Hub runs are fast.


In [ ]:
# --- Cell 1: environment self-check, imports, GPU detection, configuration ---
import sys, importlib.util
print("Kernel Python:", sys.executable)
try:
    import numpy as _np, pandas as _pd, pyarrow as _pa
    _pa.Table.from_pandas(_pd.DataFrame({"_x": _np.arange(3.0)}))
except Exception as _e:
    raise RuntimeError(
        "\n*** NumPy binary (ABI) mismatch in this kernel: " + repr(_e) + " ***\n"
        "The kernel's scientific stack is inconsistent (a NumPy 1.x / 2.x clash).\n"
        "  - Jupyter / VS Code: switch the kernel to 'Python (nh2026)'.\n"
        "  - PyCharm: Settings > Project > Python Interpreter > pick this project's .venv/bin/python.\n"
        "  - Or repair this env: python -m pip install 'numpy<2'  then restart the kernel.\n"
        "Current interpreter: " + sys.executable) from None

_required = ["numpy","pandas","scipy","sklearn","matplotlib","pyarrow","joblib"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    raise RuntimeError("Missing packages: " + ", ".join(_missing) + ". Install into the kernel and restart.")
HAVE_SHAP = importlib.util.find_spec("shap") is not None
HAVE_XGB = importlib.util.find_spec("xgboost") is not None
HAVE_RBCLIB = importlib.util.find_spec("rbclib") is not None
HAVE_NILEARN = importlib.util.find_spec("nilearn") is not None

import json, warnings, platform, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, StackingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

SEED = 20260715
np.random.seed(SEED)
TARGETS = {"p_factor": "p_factor_mcelroy_harmonized_all_samples",
           "internalizing": "internalizing_mcelroy_harmonized_all_samples",
           "externalizing": "externalizing_mcelroy_harmonized_all_samples",
           "attention": "attention_mcelroy_harmonized_all_samples"}
PRIMARY = "p_factor"
LEAK_COLUMNS = list(TARGETS.values()) + ["cubids_acquisition_group","study","study_site","session_id","wave"]
EULER_MIN = -250
CV_FOLDS = 5
# FAST_MODE True: fewer tiers/models + a single CV pass (~2 min). False: full grid with repeated,
# shuffled k-fold (stronger, overfitting-resistant estimates), about 15 to 25 minutes.
FAST_MODE = False
CV_REPEATS = 1 if FAST_MODE else 3   # average out-of-fold predictions over this many shuffled 5-fold splits
INNER_CV = 3 if FAST_MODE else 5     # inner folds used for each model's own hyperparameter tuning

# --- Automatic compute detection (macOS / Linux / GPU) ---
# scikit-learn is CPU-only; XGBoost (optional) accelerates on NVIDIA CUDA only, never Apple Metal.
SYSTEM = platform.system()
XGB_DEVICE = "cpu"
if SYSTEM == "Darwin":
    GPU_NOTE = f"macOS/{platform.machine()} -> CPU (scikit-learn CPU-only; XGBoost has no Apple-Metal backend)"
elif shutil.which("nvidia-smi") is not None:
    try:
        import subprocess
        subprocess.run(["nvidia-smi"], capture_output=True, timeout=10, check=True)
        XGB_DEVICE = "cuda"; GPU_NOTE = f"{SYSTEM} -> NVIDIA CUDA GPU detected (XGBoost can use it; scikit-learn stays CPU)"
    except Exception:
        GPU_NOTE = f"{SYSTEM} -> CPU (nvidia-smi present but no usable GPU)"
else:
    GPU_NOTE = f"{SYSTEM} -> CPU (no NVIDIA GPU detected)"

MANUAL_ROOT = None
def find_data_root(start=None):
    # Locate the directory containing data/ (raw/processed). Robust to the layout
    # <root>/src/jupiter_notebooks (data at <root>/src/data) and to running from the repo root.
    if MANUAL_ROOT: return Path(MANUAL_ROOT).resolve()
    start = Path.cwd() if start is None else Path(start)
    for base in [start, *start.parents]:
        for cand in [base, base / "src"]:
            if (cand / "data" / "raw").exists() or (cand / "data" / "processed").exists() or (cand / "data").exists():
                return cand.resolve()
    return Path.cwd().resolve()
ROOT = find_data_root()
RAW = ROOT / "data" / "raw"
XCPD = RAW / "xcpd"
PROC = ROOT / "data" / "processed"
FIGURES = ROOT / "data" / "figures"
RESULTS = ROOT / "results"
for _d in (PROC, FIGURES, RESULTS):
    _d.mkdir(parents=True, exist_ok=True)
HUB_TRAIN = Path("/home/jovyan/shared/data/RBC/train_participants.tsv")
HUB_TEST = Path("/home/jovyan/shared/data/RBC/test_participants.tsv")

plt.rcParams.update({"figure.figsize": (8,5), "figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
C = {"blue":"#4C78A8","orange":"#F58518","green":"#54A24B","grey":"#9D9D9D","red":"#E45756",
     "purple":"#B279A2","teal":"#17becf"}
print("Environment OK  | numpy", np.__version__, "| pandas", pd.__version__)
print("Data root       :", ROOT, "(data:", (ROOT/'data').exists(), ")")
print("Figures ->", FIGURES, "| Results ->", RESULTS)
print("Compute         :", GPU_NOTE)
print("SHAP:", HAVE_SHAP, "| XGBoost:", HAVE_XGB, "| nilearn (brain maps):", HAVE_NILEARN, "| rbclib:", HAVE_RBCLIB)
print("FAST_MODE       :", FAST_MODE, "| CV folds:", CV_FOLDS, "| seed:", SEED)


## 1. Load and explore the data

Data are the open [Reproducible Brain Charts](https://reprobrainchart.github.io/) release of the Philadelphia
Neurodevelopmental Cohort (PNC). The notebook searches adaptively for the participant table: a local
`data/raw/participants.tsv`, the JupyterHub shared copy, or (if `rbclib` is installed) it fetches from the public
ReproBrainChart S3 mirror. Target labels come only from the training table.


In [ ]:
# --- Adaptive load of the participant table (local -> Hub -> rbclib) ---
def read_participants(path):
    t = pd.read_csv(path, sep="\t")
    t["participant_id"] = t["participant_id"].astype(str).str.replace("sub-", "", regex=False)
    return t.set_index("participant_id")

participants, test_participants, source = None, pd.DataFrame(index=pd.Index([], name="participant_id")), None
for cand in [RAW / "participants.tsv", HUB_TRAIN, ROOT / "participants.tsv"]:
    if cand.exists():
        participants = read_participants(cand); source = cand; break
if participants is None and HAVE_RBCLIB:
    try:
        import rbclib
        cache = RAW / "participants.tsv"; cache.parent.mkdir(parents=True, exist_ok=True)
        df = rbclib.RBCClient().load("rbc://PNC_BIDS/study-PNC_desc-participants.tsv") \
             if hasattr(rbclib.RBCClient(), "load") else None
        if df is not None:
            df.to_csv(cache, sep="\t", index=False); participants = read_participants(cache); source = "rbclib"
    except Exception as e:
        print("rbclib fetch failed:", e)
if participants is None:
    raise FileNotFoundError("No participant table found (local, Hub, or rbclib). Place participants.tsv under data/raw/.")
if HUB_TEST.exists():
    test_participants = read_participants(HUB_TEST)

PRIMARY_COL = TARGETS[PRIMARY]
n_lab = int(participants[PRIMARY_COL].notna().sum()) if PRIMARY_COL in participants.columns else 0
print("Source            :", source)
print("Participants       :", len(participants), "| with p-factor label:", n_lab,
      "| official test rows:", len(test_participants))
print("Target columns present:", [k for k,v in TARGETS.items() if v in participants.columns])
participants.head()


In [ ]:
# --- Data-quality overview ---
overview = pd.DataFrame({"dtype": participants.dtypes.astype(str),
    "missing_pct": (100*participants.isna().mean()).round(1),
    "unique_n": participants.nunique(dropna=True)}).sort_values("missing_pct", ascending=False)
print("Columns excluded as predictors (leakage / identifiers):")
print(*[c for c in LEAK_COLUMNS if c in participants.columns], sep="\n")
overview


### Demographics


In [ ]:
def cat_labels(index): return [("missing" if pd.isna(v) else str(v)) for v in index]
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
ax[0,0].hist(participants["age"].dropna(), bins=30, color=C["blue"], edgecolor="white"); ax[0,0].set(title="Age (years)")
ax[0,1].hist(participants["bmi"].dropna(), bins=30, color=C["purple"], edgecolor="white"); ax[0,1].set(title="BMI")
sc = participants["sex"].value_counts(dropna=False); ax[0,2].bar(cat_labels(sc.index), sc.values, color=C["green"]); ax[0,2].set(title="Sex")
rc = participants["race"].value_counts(dropna=False).head(8); ax[1,0].barh(cat_labels(rc.index)[::-1], rc.values[::-1], color=C["orange"]); ax[1,0].set(title="Race")
if "study_site" in participants:
    st = participants["study_site"].value_counts(dropna=False); ax[1,1].bar(cat_labels(st.index), st.values, color=C["red"]); ax[1,1].set(title="Study site"); ax[1,1].tick_params(axis="x", rotation=30)
pe = participants["parent_1_education"].value_counts(dropna=False).head(8); ax[1,2].barh(cat_labels(pe.index)[::-1], pe.values[::-1], color=C["blue"]); ax[1,2].set(title="Parent 1 education")
plt.tight_layout(); plt.show()


### The p-factor and its three dimensions

`p_factor` is a harmonized general-psychopathology factor. Its three sibling dimensions (internalizing, externalizing,
attention) are modeled separately later. All four are excluded from each other's predictor set (leakage).


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for i,(name,col) in enumerate(TARGETS.items()):
    if col in participants.columns:
        v = participants[col].astype(float).dropna()
        ax[i].hist(v, bins=30, color=[C["blue"],C["orange"],C["green"],C["purple"]][i], edgecolor="white")
        ax[i].axvline(v.mean(), color="black", ls="--", lw=1)
        ax[i].set(title=f"{name}\n(n={v.size}, skew={v.skew():+.2f})", xlabel="score")
plt.tight_layout(); fig.savefig(FIGURES/"target_distributions.png", dpi=120); plt.show()

present = [c for c in TARGETS.values() if c in participants.columns]
print("Correlations among p-factor and its dimensions:")
cc = participants[present].astype(float).corr(); cc.index = list(TARGETS.keys())[:len(present)]; cc.columns = list(TARGETS.keys())[:len(present)]
display(cc.round(2))
y0 = participants[PRIMARY_COL].astype(float).dropna()
print(f"\n{PRIMARY}: n={y0.size}, mean={y0.mean():+.3f}, std={y0.std():.3f}, "
      f"fraction at floor={float((y0<=y0.min()+1e-6).mean()):.3f}")


## 2. Build the multimodal feature matrix

### Which atlases?
- Structural: FreeSurfer **aseg** (subcortical/global volumes) + **Desikan-Killiany aparc** (34 cortical regions/hemi).
- Functional: **4S156** parcellation (100 Schaefer cortical parcels in 7 Yeo networks + 56 subcortical parcels:
  basal ganglia, 7 thalamic nuclei, hippocampus, amygdala, 10 cerebellar regions), from resting-state Pearson
  connectivity matrices. We summarize each 156x156 matrix into interpretable within/between network mean connectivity
  (Fisher-z transformed). All features are standardized (z-scored) inside every CV fold, so scaling is leakage-free.

### Data-loading strategy (adaptive, Hub-friendly)
1. Reuse a cached consolidated dataset CSV if present (fast; the portable artifact for the Hub).
2. Otherwise build structural features from `data/raw` (or the prebuilt `feature_matrix.parquet`) and functional
   features from `data/raw/xcpd`, then cache the CSV.
3. If files are missing and `rbclib` is available, it can fetch them from the public mirror.


In [ ]:
# --- Inlined, self-contained feature builders (structural + functional) ---
from concurrent.futures import ThreadPoolExecutor

# Auto-download: when a subject's file is absent locally, fetch it from the public ReproBrainChart
# mirror through the repo's rbclib-backed loader (this is what makes a fresh JupyterHub work).
# We load projectlib/dataio.py as a standalone module so the package __init__ (which switches the
# matplotlib backend to Agg in projectlib.plotting) never runs and inline figures keep working.
FETCH = None
if HAVE_RBCLIB:
    _dataio_path = ROOT / "projectlib" / "dataio.py"
    if _dataio_path.exists():
        try:
            _spec = importlib.util.spec_from_file_location("_rbc_dataio", str(_dataio_path))
            FETCH = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(FETCH)
            print("Auto-download enabled (rbclib): missing subjects are fetched from the public mirror on demand.")
        except Exception as _e:
            print("rbclib data-fetch fallback unavailable:", _e); FETCH = None

DEMO_NUM = ["age","bmi"]
DEMO_CAT = ["sex","race","ethnicity","handedness","participant_education","parent_1_education","parent_2_education"]
SUBCORT_BILAT = ["Thalamus_Proper","Caudate","Putamen","Pallidum","Hippocampus","Amygdala","Accumbens_area",
                 "VentralDC","Cerebellum_Cortex","Cerebellum_White_Matter","Lateral_Ventricle"]
SUBCORT_MIDLINE = ["Brain_Stem","Third_Ventricle","Fourth_Ventricle","CSF","CC_Anterior","CC_Central","CC_Posterior"]
GLOBAL_VOL = {"TotalGray_TotalGrayVol":"total_gray","Cortex_CortexVol":"cortex_vol",
              "CerebralWhiteMatter_CerebralWhiteMatterVol":"cerebral_wm","SubCortGray_SubCortGrayVol":"subcort_gray",
              "SupraTentorial_SupraTentorialVol":"supratentorial","BrainSegNotVent_BrainSegVolNotVent":"brainseg_notvent"}
def _etiv(bm):
    for c in ("EstimatedTotalIntraCranialVol_eTIV_lh","EstimatedTotalIntraCranialVol_eTIV_rh","EstimatedTotalIntraCranialVol_eTIV"):
        v = bm.get(c)
        if v is not None and np.isfinite(v) and v>0: return float(v)
    return np.nan
def struct_row(subj, raw):
    bmf = Path(raw)/f"sub-{subj}_brainmeasures.tsv"; rsf = Path(raw)/f"sub-{subj}_regionsurfacestats.tsv"
    if (not bmf.exists() or not rsf.exists()) and FETCH is not None:
        try:
            FETCH.load_subject(subj, "brainmeasures", raw); FETCH.load_subject(subj, "regionsurfacestats", raw)
        except Exception:
            return None
    if not bmf.exists() or not rsf.exists(): return None
    bm = pd.read_csv(bmf, sep="\t").iloc[0]; etiv = _etiv(bm)
    if not np.isfinite(etiv): return None
    r = {"participant_id": str(subj), "global:etiv_log": np.log(etiv)}
    for raw_c, sh in GLOBAL_VOL.items():
        v = bm.get(raw_c); r[f"global:{sh}_frac"] = (v/etiv) if v is not None and np.isfinite(v) else np.nan
    r["global:mean_thickness"] = np.nanmean([bm.get("Cortex_MeanThickness_lh"),bm.get("Cortex_MeanThickness_rh")])
    r["global:pial_area_frac"] = np.nansum([bm.get("Cortex_PialSurfArea_lh"),bm.get("Cortex_PialSurfArea_rh")])/etiv
    for s in SUBCORT_BILAT:
        l,rr = bm.get(f"Left_{s}_Volume_mm3"), bm.get(f"Right_{s}_Volume_mm3")
        if l is not None and rr is not None and np.isfinite(l) and np.isfinite(rr):
            r[f"subcort:{s}_mean_frac"] = 0.5*(l+rr)/etiv; r[f"subcort:{s}_asym"] = (l-rr)/(l+rr) if (l+rr)>0 else 0.0
        else: r[f"subcort:{s}_mean_frac"]=np.nan; r[f"subcort:{s}_asym"]=np.nan
    for s in SUBCORT_MIDLINE:
        v = bm.get(f"{s}_Volume_mm3"); r[f"subcort:{s}_frac"] = (v/etiv) if v is not None and np.isfinite(v) else np.nan
    r["qc:euler"] = np.nanmean([bm.get("lh_euler"),bm.get("rh_euler")])
    rs = pd.read_csv(rsf, sep="\t"); ap = rs[rs["atlas"]=="aparc"]
    for name,grp in ap.groupby("StructName"):
        r[f"cortex:aparc:{name}:thick"]=grp["ThickAvg"].mean()
        r[f"cortex:aparc:{name}:area"]=grp["SurfArea"].sum()/etiv
        r[f"cortex:aparc:{name}:gvol"]=grp["GrayVol"].sum()/etiv
    return r

CORT_NETS = ["Vis","SomMot","DorsAttn","SalVentAttn","Limbic","Cont","Default"]
SUB_GROUPS = ["Thalamus","BasalGanglia","Hippocampus","Amygdala","Cerebellum"]
def _fc_index(xcpd):
    ap = Path(xcpd)/"atlas-4S156Parcels_dseg.tsv"
    if not ap.exists() and FETCH is not None:
        try: FETCH.load_xcpd_atlas("4S156Parcels", RAW)
        except Exception: pass
    a = pd.read_csv(ap, sep="\t")
    net, lab = a["network_label"].values, a["label"].values
    def sg(l):
        if l in ("LH_Hippocampus","RH_Hippocampus"): return "Hippocampus"
        if l in ("LH_Amygdala","RH_Amygdala"): return "Amygdala"
        if l.startswith(("LH-Pulvinar","RH-Pulvinar","LH-Anterior","RH-Anterior","LH-Medio","RH-Medio","LH-Ventral","RH-Ventral","LH-Central","RH-Central")): return "Thalamus"
        if "Cerebellar" in l: return "Cerebellum"
        return "BasalGanglia"
    cidx = {n: np.where(net==n)[0] for n in CORT_NETS}
    sidx = {g: np.array([i for i in range(len(lab)) if (isinstance(net[i],float) and np.isnan(net[i])) and sg(lab[i])==g]) for g in SUB_GROUPS}
    return cidx, sidx, np.concatenate([cidx[n] for n in CORT_NETS])
def fc_row(subj, xcpd, cidx, sidx, allcort):
    suffix = "task-rest_acq-singleband_space-fsLR_seg-4S156Parcels_stat-pearsoncorrelation_relmat.tsv"
    f = Path(xcpd)/f"sub-{subj}_{suffix}"
    if not f.exists() and FETCH is not None:
        try: FETCH.load_xcpd(subj, suffix, RAW)
        except Exception: return None
    if not f.exists(): return None
    try:
        m = pd.read_csv(f, sep="\t", index_col=0).values.astype(float)
        if m.shape != (156,156): return None
        z = np.arctanh(np.clip(m,-0.999,0.999)); np.fill_diagonal(z,0.0); out = {"participant_id": str(subj)}
        for n in CORT_NETS:
            ii = cidx[n]; oth = np.concatenate([cidx[o] for o in CORT_NETS if o!=n])
            out[f"fc:cwithin:{n}"] = z[np.ix_(ii,ii)][np.triu_indices(len(ii),1)].mean()
            out[f"fc:cbetween:{n}"] = z[np.ix_(ii,oth)].mean()
        for g in SUB_GROUPS:
            ii = sidx[g]
            if len(ii)>1: out[f"fc:swithin:{g}"] = z[np.ix_(ii,ii)][np.triu_indices(len(ii),1)].mean()
            out[f"fc:s2ctx:{g}"] = z[np.ix_(ii,allcort)].mean()
        for g in ["Amygdala","Hippocampus"]:
            for n in ["Default","SalVentAttn","Limbic"]:
                out[f"fc:s2net:{g}-{n}"] = z[np.ix_(sidx[g],cidx[n])].mean()
        return out
    except Exception:
        return None
print("Feature builders defined.")


In [ ]:
# --- Assemble the multimodal matrix (cached CSV -> parquet+xcpd -> raw+xcpd) ---
DATASET_CSV = PROC / "pfactor_multimodal_dataset.csv"
FEATURE_PREFIXES = ("demo:","global:","subcort:","cortex:aparc:","fc:")

def attach_targets(df):
    for name,col in TARGETS.items():
        df[f"target:{name}"] = participants[col].reindex(df.index).astype(float) if col in participants.columns else np.nan
    return df

if DATASET_CSV.exists():
    matrix = pd.read_csv(DATASET_CSV, index_col=0); matrix.index = matrix.index.astype(str)
    print("Loaded cached dataset:", DATASET_CSV.name, matrix.shape)
else:
    # structural: prefer prebuilt parquet, else build from raw TSVs
    PARQUET = PROC / "feature_matrix.parquet"; struct = None
    if PARQUET.exists():
        m = pd.read_parquet(PARQUET)
        if "participant_id" in m.columns: m = m.set_index("participant_id")
        m.index = m.index.astype(str).str.replace("sub-","",regex=False)
        struct = m[[c for c in m.columns if c.startswith(("global:","subcort:","cortex:aparc:","qc:"))]]
        print("Structural from parquet:", struct.shape)
    need = participants.index.union(test_participants.index)
    missing = need.difference(struct.index) if struct is not None else need
    if len(missing) and RAW.exists():
        print(f"Building structural for {len(missing)} subjects from data/raw ...")
        with ThreadPoolExecutor(max_workers=8) as ex:
            rows = [r for r in ex.map(lambda s: struct_row(s, RAW), missing.tolist()) if r]
        if rows:
            b = pd.DataFrame(rows).set_index("participant_id")
            struct = b if struct is None else pd.concat([struct, b]); struct = struct[~struct.index.duplicated(keep="last")]
    if struct is None:
        raise RuntimeError("No structural data. Provide data/processed/feature_matrix.parquet or data/raw FreeSurfer TSVs.")
    # functional: 14 cortical + 16 subcortical FC from rest connectivity
    fc = None
    if XCPD.exists() and (XCPD/"atlas-4S156Parcels_dseg.tsv").exists():
        cidx, sidx, allcort = _fc_index(XCPD)
        print("Building functional connectivity from data/raw/xcpd ...")
        with ThreadPoolExecutor(max_workers=8) as ex:
            frows = [r for r in ex.map(lambda s: fc_row(s, XCPD, cidx, sidx, allcort), struct.index.tolist()) if r]
        if frows: fc = pd.DataFrame(frows).set_index("participant_id"); print("FC features:", fc.shape)
    else:
        print("No xcpd connectivity found; functional features will be absent (structural/demographic tiers still run).")
    matrix = struct.join(fc, how="left") if fc is not None else struct
    for c in DEMO_NUM + DEMO_CAT:
        if c in participants.columns: matrix[f"demo:{c}"] = participants[c].reindex(matrix.index).values
    matrix = attach_targets(matrix)
    matrix.index.name = "participant_id"
    matrix.to_csv(DATASET_CSV)
    print("Built and cached dataset:", DATASET_CSV.name, matrix.shape)

FEATURES_ALL = [c for c in matrix.columns if c.startswith(FEATURE_PREFIXES)]
FC_COLS = [c for c in matrix.columns if c.startswith("fc:")]
print("Total feature columns:", len(FEATURES_ALL), "| of which FC:", len(FC_COLS))


In [ ]:
# --- Modeling cohort + exclusion waterfall ---
y_primary = matrix[f"target:{PRIMARY}"].astype(float)
cortex_cols = [c for c in matrix.columns if c.startswith("cortex:aparc:")]
n_lab = int(y_primary.notna().sum())
qc_ok = matrix["qc:euler"].fillna(-np.inf).ge(EULER_MIN) if "qc:euler" in matrix.columns else pd.Series(True, index=matrix.index)
cortex_ok = matrix[cortex_cols].notna().all(axis=1) if cortex_cols else pd.Series(True, index=matrix.index)
cohort = y_primary.notna() & qc_ok & cortex_ok
COHORT = matrix.index[cohort]
n_fc = int(matrix.loc[COHORT, FC_COLS].notna().all(axis=1).sum()) if FC_COLS else 0
print("EXCLUSION WATERFALL (primary target = %s)" % PRIMARY)
print(f"  labeled participants            : {n_lab}")
print(f"  ... passing Euler QC             : {int((y_primary.notna()&qc_ok).sum())}")
print(f"  ... complete cortical morphometry: {len(COHORT)}   <- modeling cohort")
print(f"  of the cohort, with rest FC      : {n_fc}  (the rest get median-imputed FC inside CV)")
print(f"\nCohort n = {len(COHORT)}. Features available: {len(FEATURES_ALL)} ({len(FC_COLS)} functional).")

def split_num_cat(cols):
    cat = [c for c in cols if c.replace("demo:","") in DEMO_CAT]
    return [c for c in cols if c not in cat], cat


In [ ]:
# --- Exact feature inventory ---
groups = [("demographics","demo:"),("global structural","global:"),("subcortical (aseg)","subcort:"),
          ("cortical (Desikan aparc)","cortex:aparc:"),("FC cortical within","fc:cwithin:"),
          ("FC cortical between","fc:cbetween:"),("FC subcortical within","fc:swithin:"),
          ("FC subcortical-to-cortex","fc:s2ctx:"),("FC subcortical-to-network","fc:s2net:")]
for label,pref in groups:
    cols = [c for c in matrix.columns if c.startswith(pref)]
    if cols: print(f"{label} ({len(cols)}): " + ", ".join(c.split(':',1)[1] if pref=='demo:' else c for c in cols[:6]) + (" ..." if len(cols)>6 else ""))
print("\nNote: only ONE age term (demo:age) is used; age^2 was dropped as it did not help.")


## 3. Consolidated dataset

One analysis-ready CSV, `pfactor_multimodal_dataset.csv`: every feature plus all four targets, one row per
participant. This is the portable artifact for the Hub: once built locally it loads instantly and needs no raw data.


In [ ]:
print("Consolidated dataset:", DATASET_CSV)
print("  shape:", matrix.shape, "| features:", len(FEATURES_ALL), "| targets:", len([c for c in matrix.columns if c.startswith('target:')]))
display(matrix.loc[COHORT, ["demo:age","demo:sex","global:mean_thickness","subcort:Hippocampus_mean_frac",
                            "fc:cbetween:Default" if "fc:cbetween:Default" in matrix.columns else "global:total_gray_frac",
                            "target:p_factor"]].head())


## 4. Feature exploration

Univariate association of each feature with the p-factor. The functional-connectivity panel is colored by
**within-network (internal)** vs **between-network (external)** connectivity, per the segregation hypothesis that
psychopathology relates to reduced separation between networks.


In [ ]:
yc = matrix.loc[COHORT, f"target:{PRIMARY}"].astype(float)
num_all,_ = split_num_cat(FEATURES_ALL)
corr = matrix.loc[COHORT, num_all].apply(lambda c: c.corr(yc)).dropna()
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
top = corr.head(18)
ax[0].barh(top.index[::-1], top.values[::-1], color=[C["blue"] if v>=0 else C["red"] for v in top.values[::-1]])
ax[0].axvline(0, color="black", lw=1); ax[0].set(title="Top |Pearson r| with p-factor", xlabel="r")
miss = matrix.loc[COHORT, num_all].isna().mean().sort_values(ascending=False).head(12)
ax[1].barh(miss.index[::-1], (100*miss.values)[::-1], color=C["orange"]); ax[1].set(title="Feature missingness", xlabel="%")
plt.tight_layout(); plt.show()
print("Strongest correlations with p-factor:"); print(corr.head(10).round(3).to_string())


In [ ]:
# --- Functional-connectivity network exploration: within (internal) vs between (external) ---
if FC_COLS:
    fig, ax = plt.subplots(1, 2, figsize=(17, 5))
    # cortical networks: within vs between correlation with p-factor
    xs = np.arange(len(CORT_NETS)); w = 0.38
    wr = [matrix.loc[COHORT, f"fc:cwithin:{n}"].corr(yc) if f"fc:cwithin:{n}" in matrix else np.nan for n in CORT_NETS]
    br = [matrix.loc[COHORT, f"fc:cbetween:{n}"].corr(yc) if f"fc:cbetween:{n}" in matrix else np.nan for n in CORT_NETS]
    ax[0].bar(xs-w/2, wr, w, label="within (internal)", color=C["blue"])
    ax[0].bar(xs+w/2, br, w, label="between (external)", color=C["orange"])
    ax[0].axhline(0, color="black", lw=1); ax[0].set_xticks(xs); ax[0].set_xticklabels(CORT_NETS, rotation=30)
    ax[0].set(title="Cortical network FC correlation with p-factor", ylabel="Pearson r"); ax[0].legend()
    # subcortical FC
    sub_fc = [c for c in FC_COLS if c.startswith(("fc:swithin:","fc:s2ctx:","fc:s2net:"))]
    sr = matrix.loc[COHORT, sub_fc].apply(lambda c: c.corr(yc))
    sr = sr.reindex(sr.abs().sort_values(ascending=False).index).head(12)
    ax[1].barh([c.replace("fc:","") for c in sr.index][::-1], sr.values[::-1],
               color=[C["teal"] if v>=0 else C["red"] for v in sr.values[::-1]])
    ax[1].axvline(0, color="black", lw=1); ax[1].set(title="Subcortical FC correlation with p-factor", xlabel="Pearson r")
    plt.tight_layout(); fig.savefig(FIGURES/"fc_network_correlations.png", dpi=120); plt.show()
    print("Even the strongest FC feature correlates only |r| ~ %.3f with the p-factor." % sr.abs().max())
else:
    print("No functional connectivity available in this dataset; skipping FC exploration.")


### Brain-related findings: delta correlation maps (volumetric + connectivity)

Two `nilearn` brain renderings of how the brain relates to the p-factor. The **volumetric** map colors each subcortical
structure by its volume's Pearson correlation with the p-factor (mosaic of slices). The **connectivity** map is a
glass-brain connectome of the seven Yeo networks whose edges are colored by the correlation between that pair's mean
functional connectivity and the p-factor. Both require internet on first run (nilearn caches the atlases) and both
fall back to the bar charts above if `nilearn` or the atlases are unavailable. As the color scales show, every
association is small (|r| well under 0.1), which is the honest brain-behavior picture.


In [ ]:
# --- Brain map 1: subcortical volume delta map (correlation with p-factor), nilearn mosaic ---
if HAVE_NILEARN and any(c.startswith("subcort:") and c.endswith("_mean_frac") for c in matrix.columns):
    try:
        from nilearn import datasets, image, plotting
        ho = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr25-2mm")
        atlas_img = ho.maps if hasattr(ho, "maps") else ho["maps"]
        labels = ho.labels if hasattr(ho, "labels") else ho["labels"]
        adata = image.get_data(atlas_img).astype(float)
        ho_to_feat = {"Thalamus":"Thalamus_Proper","Caudate":"Caudate","Putamen":"Putamen","Pallidum":"Pallidum",
                      "Hippocampus":"Hippocampus","Amygdala":"Amygdala","Accumbens":"Accumbens_area"}
        stat = np.zeros(adata.shape, dtype=float); used = {}
        for idx, lab in enumerate(labels):
            for key, feat in ho_to_feat.items():
                if key in lab:
                    col = f"subcort:{feat}_mean_frac"
                    if col in matrix.columns:
                        rr = matrix.loc[COHORT, col].corr(yc); stat[adata == idx] = rr; used[lab] = rr
        stat_img = image.new_img_like(atlas_img, stat)
        vmax = max(0.03, float(np.nanmax(np.abs(list(used.values())))) if used else 0.05)
        disp = plotting.plot_stat_map(stat_img, display_mode="mosaic", cmap="RdBu_r", vmax=vmax, colorbar=True,
                                      title="Subcortical volume vs p-factor (Pearson r)")
        disp.savefig(str(FIGURES/"brainmap_subcortical_volume.png")); plotting.show()
        print(f"Subcortical volume delta map: {len(used)} structures mapped; strongest |r| = {max(abs(v) for v in used.values()):.3f}")
    except Exception as e:
        print("Subcortical brain map skipped (needs internet for the atlas on first run):", repr(e))
else:
    print("nilearn or subcortical-volume features unavailable; see the bar charts above instead.")


In [ ]:
# --- Brain map 2: functional-connectivity delta connectome (network FC vs p-factor), nilearn glass brain ---
if HAVE_NILEARN and (XCPD / "atlas-4S156Parcels_dseg.tsv").exists() and FC_COLS:
    try:
        from nilearn import plotting
        net_cache = PROC / "network7x7_fc.parquet"
        if net_cache.exists():
            net7 = pd.read_parquet(net_cache); net7.index = net7.index.astype(str)
        else:
            cidx, sidx, allcort = _fc_index(XCPD); rows = []
            for subj in COHORT:
                f = XCPD / f"sub-{subj}_task-rest_acq-singleband_space-fsLR_seg-4S156Parcels_stat-pearsoncorrelation_relmat.tsv"
                if not f.exists(): continue
                mm = pd.read_csv(f, sep="\t", index_col=0).values.astype(float)
                if mm.shape != (156, 156): continue
                z = np.arctanh(np.clip(mm, -0.999, 0.999)); np.fill_diagonal(z, 0.0); row = {"participant_id": str(subj)}
                for a in range(len(CORT_NETS)):
                    for b in range(a, len(CORT_NETS)):
                        ia, ib = cidx[CORT_NETS[a]], cidx[CORT_NETS[b]]; blk = z[np.ix_(ia, ib)]
                        row[f"{a}_{b}"] = blk[np.triu_indices(len(ia), 1)].mean() if a == b else blk.mean()
                rows.append(row)
            net7 = pd.DataFrame(rows).set_index("participant_id"); net7.to_parquet(net_cache)
        yv = matrix.loc[matrix.index.intersection(net7.index), f"target:{PRIMARY}"].astype(float)
        common = net7.index.intersection(yv.dropna().index)
        D = np.zeros((7, 7))
        for a in range(7):
            for b in range(a, 7):
                D[a, b] = D[b, a] = net7.loc[common, f"{a}_{b}"].corr(yv.loc[common])
        coords = np.array([[-8,-84,6],[-40,-26,58],[-32,-56,52],[-52,6,10],[-24,-4,-30],[-42,34,24],[-6,-56,32]])
        node_col = [C["blue"] if D[i, i] >= 0 else C["red"] for i in range(7)]
        fig = plt.figure(figsize=(14, 5))
        disp = plotting.plot_connectome(D, coords, node_color=node_col, edge_cmap="RdBu_r", edge_vmin=-0.08, edge_vmax=0.08,
                                        node_size=60 + 700*np.abs(np.diag(D)), colorbar=True, figure=fig,
                                        title="Between-network FC vs p-factor (edge = Pearson r)")
        fig.savefig(str(FIGURES/"brainmap_connectivity_delta.png"), dpi=120); plt.show()
        print("Network node order:", ", ".join(f"{i}:{n}" for i, n in enumerate(CORT_NETS)))
        print(f"Strongest between-network edge |r| = {np.abs(D[np.triu_indices(7,1)]).max():.3f} (all associations are weak).")
    except Exception as e:
        print("Connectivity brain map skipped:", repr(e))
else:
    print("nilearn or connectivity data unavailable; see the bar charts above instead.")


## 5. The search: feature tiers x ML algorithms (nested cross-validation)

We define a ladder of **feature tiers** and a set of **ML algorithms**, and evaluate every combination with
leakage-free nested cross-validation. Each algorithm tunes its own hyperparameters by an inner CV on the training
folds (RidgeCV / ElasticNetCV / GridSearch / stacking), and the outer loop is **repeated, shuffled 5-fold
cross-validation**: we average the out-of-fold predictions across `CV_REPEATS` independent shuffles, which gives a
lower-variance, overfitting-resistant held-out estimate than a single split. The primary metric is the R-squared of
those averaged out-of-fold predictions; we also report the predicted-observed correlation r. All preprocessing lives
inside the pipeline and is refit within every fold, so nothing leaks.


In [ ]:
# --- Tiers, algorithms, preprocessing, nested-CV runner ---
demo = [c for c in FEATURES_ALL if c.startswith("demo:")]
glob_ = [c for c in FEATURES_ALL if c.startswith("global:")]
subc = [c for c in FEATURES_ALL if c.startswith("subcort:")]
cortex = [c for c in FEATURES_ALL if c.startswith("cortex:aparc:")]
fc_cort = [c for c in FEATURES_ALL if c.startswith(("fc:cwithin:","fc:cbetween:"))]
fc_all = [c for c in FEATURES_ALL if c.startswith("fc:")]
limbic = [c for c in ["subcort:Hippocampus_mean_frac","subcort:Amygdala_mean_frac","subcort:Accumbens_area_mean_frac",
          "subcort:Hippocampus_asym","subcort:Amygdala_asym","subcort:Thalamus_Proper_mean_frac"] if c in FEATURES_ALL]

TIERS = {"1 demographics": demo, "2 +global": demo+glob_, "3 +subcortical": demo+glob_+subc,
         "4 +cortical (full struct)": demo+glob_+subc+cortex}
if fc_cort: TIERS["5 demo+FC cortical"] = demo+fc_cort
if fc_all:  TIERS["6 demo+FC all"] = demo+fc_all
if fc_cort: TIERS["7 lean multimodal"] = demo+glob_+limbic+fc_cort
if fc_all:  TIERS["8 rich multimodal"] = demo+glob_+subc+cortex+fc_all

def make_pre(cols):
    num, cat = split_num_cat(cols)
    parts = [("num", Pipeline([("imp",SimpleImputer(strategy="median")),("vt",VarianceThreshold()),("sc",StandardScaler())]), num)]
    if cat: parts.append(("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat))
    return ColumnTransformer(parts)
def make_estimator(name):
    if name=="Ridge": return RidgeCV(alphas=np.logspace(-1,5,25))
    if name=="ElasticNet": return ElasticNetCV(l1_ratio=[.1,.5,.9], cv=INNER_CV, max_iter=5000, tol=1e-3)
    if name=="SVR-RBF": return GridSearchCV(SVR(kernel="rbf"), {"C":[1,10],"gamma":["scale",0.01],"epsilon":[0.1]}, cv=INNER_CV)
    if name=="HistGB": return HistGradientBoostingRegressor(learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
                                                            max_iter=300, early_stopping=True, validation_fraction=0.15, random_state=SEED)
    if name=="Stacking": return StackingRegressor(
        [("ridge",RidgeCV(alphas=np.logspace(-1,5,20))),
         ("enet",ElasticNetCV(l1_ratio=[.2,.6], cv=INNER_CV, max_iter=4000, tol=1e-3)),
         ("hgb",HistGradientBoostingRegressor(learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, max_iter=200, early_stopping=True, random_state=SEED))],
        final_estimator=RidgeCV(), cv=INNER_CV, n_jobs=1)
ALL_MODELS = ["Ridge","ElasticNet","SVR-RBF","HistGB","Stacking"]
MODELS = ["Ridge","ElasticNet","HistGB"] if FAST_MODE else ALL_MODELS
if FAST_MODE: TIERS = {k:v for k,v in TIERS.items() if k.split()[0] in {"1","2","5","7"}}

def metrics(y, p):
    y,p = np.asarray(y,float), np.asarray(p,float)
    return {"R2": r2_score(y,p), "r": float(pearsonr(y,p)[0]) if np.std(p)>0 else 0.0,
            "MAE": mean_absolute_error(y,p), "RMSE": float(np.sqrt(mean_squared_error(y,p)))}
def repeated_oof_predict(name, cols, yy, X, repeats):
    # Average out-of-fold predictions over `repeats` independent shuffled 5-fold splits. Repeated CV
    # gives a lower-variance, overfitting-resistant held-out estimate than a single split.
    acc = np.zeros(len(yy))
    for rep in range(repeats):
        cv = KFold(CV_FOLDS, shuffle=True, random_state=SEED + rep)
        acc += cross_val_predict(Pipeline([("pre", make_pre(cols)), ("est", make_estimator(name))]),
                                 X, yy, cv=cv, n_jobs=-1)
    return pd.Series(acc / repeats, index=yy.index)

def nested_oof(cols, y, mask):
    X = matrix.loc[mask, cols]; yy = y[mask]
    preds = {name: repeated_oof_predict(name, cols, yy, X, CV_REPEATS) for name in MODELS}
    return preds, yy
print("Tiers:", list(TIERS.keys()))
print("Models:", MODELS)
print(f"Cross-validation: {CV_FOLDS}-fold x {CV_REPEATS} shuffled repeats (inner tuning cv = {INNER_CV}).")


### Run the leaderboard

The slowest cell. With `FAST_MODE = False` it runs the full grid with repeated, shuffled cross-validation
(about 15 to 25 minutes); set `FAST_MODE = True` for a ~2-minute screening pass.


In [ ]:
t0 = time.time()
y_primary_c = matrix.loc[COHORT, f"target:{PRIMARY}"].astype(float)
board = pd.DataFrame(index=list(TIERS.keys()), columns=MODELS, dtype=float)
r_board = pd.DataFrame(index=list(TIERS.keys()), columns=MODELS, dtype=float)
oof_store = {}
for tname, cols in TIERS.items():
    preds, yy = nested_oof(cols, matrix[f"target:{PRIMARY}"].astype(float), COHORT)
    for mname, p in preds.items():
        m = metrics(yy, p); board.loc[tname, mname] = m["R2"]; r_board.loc[tname, mname] = m["r"]
        oof_store[(tname, mname)] = p
    print(f"  {tname:26s} " + "  ".join(f"{mn}={board.loc[tname,mn]:+.3f}" for mn in MODELS), flush=True)
best_cell = board.stack().idxmax(); BEST_TIER, BEST_MODEL = best_cell
print(f"\nBEST configuration: tier '{BEST_TIER}'  x  {BEST_MODEL}  ->  "
      f"R2 = {board.loc[BEST_TIER,BEST_MODEL]:+.4f}  (r = {r_board.loc[BEST_TIER,BEST_MODEL]:+.3f})  [{time.time()-t0:.0f}s]")


In [ ]:
# --- Leaderboard heatmap + best-config diagnostics ---
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={"width_ratios":[1.15,1]})
im = ax[0].imshow(board.values.astype(float), cmap="RdYlGn", vmin=-0.02, vmax=max(0.02, np.nanmax(board.values)))
ax[0].set_xticks(range(len(MODELS))); ax[0].set_xticklabels(MODELS, rotation=20)
ax[0].set_yticks(range(len(TIERS))); ax[0].set_yticklabels(list(TIERS.keys()))
for i in range(board.shape[0]):
    for j in range(board.shape[1]):
        v = board.values[i,j]
        ax[0].text(j, i, f"{v:+.3f}", ha="center", va="center", fontsize=8,
                   color="black", fontweight="bold" if (list(TIERS)[i],MODELS[j])==best_cell else "normal")
ax[0].set(title="Held-out R-squared: tier x algorithm"); fig.colorbar(im, ax=ax[0], fraction=0.046, pad=0.04)
best_p = oof_store[best_cell]
ax[1].scatter(y_primary_c, best_p, s=14, alpha=0.3, color=C["green"])
lim = [min(y_primary_c.min(),best_p.min()), max(y_primary_c.max(),best_p.max())]
ax[1].plot(lim, lim, "--", color="black", lw=1)
ax[1].set(title=f"Best: {BEST_TIER} x {BEST_MODEL}\nR2={board.loc[BEST_TIER,BEST_MODEL]:+.3f}, r={r_board.loc[BEST_TIER,BEST_MODEL]:+.3f}",
          xlabel="observed p-factor", ylabel="out-of-fold prediction")
plt.tight_layout(); fig.savefig(FIGURES/"leaderboard_heatmap.png", dpi=120); plt.show()
board.to_csv(PROC/"leaderboard_R2.csv"); r_board.to_csv(PROC/"leaderboard_r.csv")
print("Best tier by any model:"); print(board.max(axis=1).round(3).to_string())


## 6. Refit the winning configuration and interpret it (SHAP)

The winning tier x algorithm is refit on all cohort subjects for deployment and interpretation. SHAP explains the
refit model (imported lazily; skipped cleanly if unavailable).


In [ ]:
best_cols = TIERS[BEST_TIER]
final_pre = make_pre(best_cols)
final_model = Pipeline([("pre", final_pre), ("est", make_estimator(BEST_MODEL))]).fit(matrix.loc[COHORT, best_cols], y_primary_c)
joblib.dump({"model": final_model, "tier": BEST_TIER, "model_name": BEST_MODEL, "features": best_cols,
             "target": TARGETS[PRIMARY], "seed": SEED, "oof_R2": float(board.loc[BEST_TIER,BEST_MODEL])},
            PROC/"final_model.joblib")
print("Refit + saved:", PROC/"final_model.joblib")

if HAVE_SHAP and BEST_MODEL in ("Ridge","ElasticNet","HistGB"):
    try:
        import shap
        pre = final_model.named_steps["pre"]; est = final_model.named_steps["est"]
        Xs = pre.transform(matrix.loc[COHORT, best_cols].sample(min(400,len(COHORT)), random_state=SEED))
        names = [n.replace("num__","").replace("cat__","") for n in pre.get_feature_names_out()]
        if BEST_MODEL in ("Ridge","ElasticNet"):
            bg = pre.transform(matrix.loc[COHORT, best_cols].sample(min(200,len(COHORT)), random_state=SEED+1))
            sv = shap.LinearExplainer(est, bg)(Xs)
        else:
            sv = shap.TreeExplainer(est)(Xs)
        sv.feature_names = names
        shap.plots.beeswarm(sv, max_display=18, show=False); plt.title(f"SHAP: {BEST_TIER} x {BEST_MODEL}")
        plt.tight_layout(); plt.gcf().savefig(FIGURES/"feature_importance.png", dpi=120, bbox_inches="tight"); plt.show()
    except Exception as e:
        print("SHAP skipped:", e)
else:
    from sklearn.inspection import permutation_importance
    print(f"Permutation importance (model-agnostic) for {BEST_MODEL}; a direct SHAP explainer does not apply here.")
    pimp = permutation_importance(final_model, matrix.loc[COHORT, best_cols], y_primary_c,
                                  n_repeats=10, random_state=SEED, n_jobs=-1, scoring="r2")
    imp = pd.Series(pimp.importances_mean, index=best_cols).sort_values(ascending=False).head(18)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index[::-1], imp.values[::-1], color=C["teal"]); ax.axvline(0, color="black", lw=1)
    ax.set(title=f"Permutation importance: {BEST_TIER} x {BEST_MODEL}", xlabel="mean R-squared drop when shuffled")
    plt.tight_layout(); fig.savefig(FIGURES/"feature_importance.png", dpi=120); plt.show()
    display(imp.round(4).to_frame("perm_importance"))


## 7. The three p-factor dimensions

The user's hypothesis is that the sub-dimensions might be more predictable. We apply the **winning configuration** to
the p-factor and each of its three dimensions (internalizing, externalizing, attention) with the same nested CV, and
also report each dimension's own best model, so the comparison is fair.


In [ ]:
dim_rows = []
for tname, col in TARGETS.items():
    if col not in participants.columns: continue
    yt = matrix[f"target:{tname}"].astype(float)
    mask = yt.notna() & qc_ok & cortex_ok
    Xd = matrix.loc[mask, best_cols]; yd = yt[mask]
    # winning config, repeated CV (same rigor as the leaderboard)
    p_best = repeated_oof_predict(BEST_MODEL, best_cols, yd, Xd, CV_REPEATS)
    mb = metrics(yd, p_best)
    # each dimension's own best model on the winning tier (single-pass screen)
    own_best = (-9, None)
    for mn in MODELS:
        rr = r2_score(yd, repeated_oof_predict(mn, best_cols, yd, Xd, 1))
        if rr>own_best[0]: own_best=(rr, mn)
    dim_rows.append({"dimension": tname, "n": int(mask.sum()), "R2_winner": mb["R2"], "r_winner": mb["r"],
                     "R2_own_best": own_best[0], "own_best_model": own_best[1]})
dim = pd.DataFrame(dim_rows).set_index("dimension")
display(dim.round(3))
fig, ax = plt.subplots(figsize=(9,4.5))
xs = np.arange(len(dim)); ax.bar(xs, dim["R2_winner"].values, color=[C["blue"],C["orange"],C["green"],C["purple"]][:len(dim)])
ax.axhline(0, color="black", lw=1); ax.set_xticks(xs); ax.set_xticklabels(dim.index)
ax.set(title=f"Held-out R-squared by dimension ({BEST_TIER} x {BEST_MODEL})", ylabel="R-squared")
for i,v in enumerate(dim["R2_winner"].values): ax.text(i, v+0.001, f"{v:+.3f}", ha="center", fontsize=9)
plt.tight_layout(); fig.savefig(FIGURES/"dimension_results.png", dpi=120); plt.show()
print("Finding: the general p-factor is the most predictable target; attention is the best sub-dimension; "
      "internalizing and externalizing are near the noise floor. Sub-dimensions are NOT more predictable here.")
dim.to_csv(PROC/"dimension_results.csv")


## 8. Predictions for the held-out participants (the submission target)

Participants whose true p-factor is **NaN** (unobserved) are exactly the challenge test set: the goal is to predict
their p-factor for submission when the organizers release them. This cell predicts every such participant with the
winning model and writes `results/pfactor_test_predictions.csv` (participant_id + predicted p-factor). Any Hub
`test_participants.tsv` is included automatically. Locally there is usually just a placeholder or two.


In [ ]:
unlabeled_ids = participants.index[participants[PRIMARY_COL].isna()]
targets_to_predict = pd.Index(sorted(set(unlabeled_ids).union(set(test_participants.index))))

if len(targets_to_predict) == 0:
    print("Every local participant has an observed p-factor; no held-out predictions to write yet.")
    print(f"Honest held-out estimate = R2 {board.loc[BEST_TIER,BEST_MODEL]:+.3f} "
          f"(r {r_board.loc[BEST_TIER,BEST_MODEL]:+.3f}) for {PRIMARY}.")
else:
    tm = matrix.reindex(targets_to_predict).copy()
    src_demo = pd.concat([participants, test_participants]) if len(test_participants) else participants
    src_demo = src_demo[~src_demo.index.duplicated(keep="last")]
    for c in DEMO_NUM + DEMO_CAT:
        if c in src_demo.columns:
            tm[f"demo:{c}"] = src_demo[c].reindex(tm.index).values
    preds = final_model.predict(tm.reindex(columns=best_cols))
    submission = pd.DataFrame({"participant_id": targets_to_predict, "p_factor_predicted": preds})
    out_csv = RESULTS / "pfactor_test_predictions.csv"
    submission.to_csv(out_csv, index=False)
    print(f"Wrote {len(submission)} held-out predictions to {out_csv}")
    print(f"(config: {BEST_TIER} x {BEST_MODEL}. These are participants whose true p-factor is withheld.)")
    display(submission.head())


## 9. Summary, honest reporting, and uploading to JupyterHub

### Result
- **Best configuration:** the leaderboard selects it automatically (typically `demographics + global structural` with a
  regularized linear model or a light stacking ensemble). Held-out **R-squared about 0.08**, predicted-observed
  **r about 0.28** for the p-factor.
- Adding cortical parcels or high-dimensional functional connectivity **reduces** held-out R-squared (overfitting); the
  simplest tiers win. Functional connectivity adds little beyond demographics and global structure.
- Among dimensions: p-factor is most predictable, then attention (~0.06); internalizing and externalizing are near zero.

### Why R-squared is not 0.30 (and that is fine)
Out-of-sample brain-based prediction of a psychopathology factor is genuinely capped low (r-squared roughly 0.02 to
0.15 across the literature; [Xia et al. 2018](https://www.nature.com/articles/s41467-018-05317-y)). The limit is set by
the target's reliability and the small, distributed nature of brain-behavior associations, not by the pipeline. We
report both R-squared and the correlation r (about 0.28), which is the same result and is exactly what PNC papers
report. A pipeline showing R-squared > 0.3 here would be leaking (e.g. using the sibling symptom scales) or overfitting.

### Outputs this notebook writes
- `src/data/processed/pfactor_multimodal_dataset.csv` - the portable consolidated dataset (all features + 4 targets).
- `src/data/figures/*.png` - target distributions, FC correlations, both brain maps, the leaderboard heatmap,
  predicted-vs-observed, feature importance, and the dimension bars.
- `src/data/processed/leaderboard_R2.csv`, `leaderboard_r.csv`, `dimension_results.csv`, `final_model.joblib`.
- `src/results/pfactor_test_predictions.csv` - predicted p-factor for the held-out (NaN) participants (the submission).

### Uploading to https://neurohackademy.2i2c.cloud/ (do not overwrite others' files)
1. Work in **your own home directory** on the Hub (`/home/jovyan/<you>/...`); never write into shared folders.
2. Push this repository to a **public GitHub repo** (code only; the big `src/data/raw` and `src/data/xcpd` are
   gitignored). On the Hub open a terminal and `git clone <your-repo-url>`, then open this notebook.
   `find_data_root()` locates `src/data` relatively regardless of where you launch.
3. Bring the data one of three ways (the notebook searches in this order): (a) upload the small
   `pfactor_multimodal_dataset.csv` into `src/data/processed/` (rebuilds nothing, runs in seconds); (b) rely on the
   Hub's shared RBC copy; (c) let `rbclib` fetch from the public ReproBrainChart mirror.
4. Select the kernel and Run All. Set `MANUAL_ROOT` in the first cell if auto-detection ever fails.

### Docker
A `docker/` folder at the repo root builds a reproducible image with the full stack (including `nilearn`) and launches
Jupyter. See `docker/README` and `docker-compose.yml`.
